# Augmentation and Generation

You learnt about the first stage of a RAG pipeline - chunking documents and storing them as embeddings in a vector store. Now, let's complete the pipeline by providing the retrieved context to a generator to get a final response.


*User Query → Retrieve → Augment → Generate → Response*

### Load the dataset

We will use a dataset specifically tailored for RAG tasks. The [RAG-Mini-Wikipedia](https://huggingface.co/datasets/rag-datasets/rag-mini-wikipedia) from Hugging Face is a curated, small-scale dataset designed to evaluate RAG pipelines. It provides ~900 questions, ground truth answers, and ~690,000 words of Wikipedia snippets for retrieving context.

Contains two main parts:
* *Passages*: A set of documents (passages) indexed for retrieval
* *QA Pairs*: ~900 questions with corresponding ground truth answers, often including difficulty ratings

To download datasets such as `rag-mini-wikipedia` from HF datasets, the standard approach is to use the `datasets` library. Using this, the data is stored in a hashed folder structure and is cached not cloned (downloaded on disk).

We have the downloaded (cloned) dataset with us. This can be done using a `git clone` command. You need to have Git installed on your system for cloning git repos.

In [ ]:
# !git clone https://huggingface.co/datasets/rag-datasets/rag-mini-wikipedia
# Stores the data in the directory where you run the command

In [ ]:
# from datasets import load_dataset
# load_dataset("rag-datasets/rag-mini-wikipedia", 'text-corpus')
# load_dataset("rag-datasets/rag-mini-wikipedia", 'question-answer')

You do not need to clone for now though. The downloaded dataset is already provided. Let's understand the dataset structure.

In [ ]:
# !tree
# or
!tree ./rag-mini-wikipedia

* Core file: `data/passages.parquet` <br>
    This is the retrieval corpus. We will perform the semantic searches for regular queries here. <br> <br>
* Query/evaluation file: `data/test.parquet` <br>
    This is the evaluation dataset containing sample questions and their (ideal) answers. We can use various metrics to check the performance of the RAG system using this data.


Note that there are two data folders here. *`data`*, which is already structured for retrieval-style tasks, and *`raw_data`*, which is the pre-processing source. We don't need the raw texts for now. 

This is not raw text; it is pre-prepared passage-level data, so your pipeline can skip chunking. However, you can try using it to add a preprocessing stage (cleaning, normalisation, chunking etc.) to the RAG flow.

Let's read the parquet data files

In [24]:
import pandas as pd, numpy as np

passages = pd.read_parquet("rag-mini-wikipedia/data/passages.parquet")
passages.head()

# we do not need the 'test' file yet
# test = pd.read_parquet("rag-mini-wikipedia/data/test.parquet")

,passage
id,
0,"Uruguay (official full name in ; pron. , Eas..."
1,"It is bordered by Brazil to the north, by Arge..."
2,Montevideo was founded by the Spanish in the e...
3,The economy is largely based in agriculture (m...
4,"According to Transparency International, Urugu..."


In [3]:
passages.shape

(3200, 1)

### Prepare Documents

Since chunking is already done, the pipeline becomes

*User Query → Embedding → Vector Search (on passages.text) → Augment prompt → LLM generates answer*

Let's prepare documents for embedding

In [7]:
documents = passages["passage"].tolist()

Optionally include metadata:

In [13]:
docs = [
    {
        "text": row["passage"],
        "id": row.name
    }
    for _, row in passages.iterrows()
]

In [14]:
docs[0]

{'text': 'Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.',
 'id': 0}

Now, let's create two vector stores, a custom one, and one using ChromaDB.

### Adding Data to Vector Store

#### Creating a Custom Vector Store

Let's use a `dict` to store the prepared documents. First we will need to encode the texts into embeddings.

In [17]:
from sentence_transformers import SentenceTransformer
embedding_fn = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
embeddings = embedding_fn.encode(documents)
embeddings

array([[ 0.00698537, -0.06149811, -0.06683703, ..., -0.00445882,
        -0.00991778,  0.01441198],
       [ 0.12249582, -0.04751833, -0.07255243, ...,  0.01259665,
         0.03551501,  0.03976375],
       [-0.03174124, -0.0598103 , -0.06112824, ..., -0.00991441,
         0.06504018, -0.02088597],
       ...,
       [ 0.05883516,  0.02168372, -0.04827357, ...,  0.03253841,
         0.04849384, -0.0205755 ],
       [ 0.07088793, -0.00946886, -0.03386718, ...,  0.01745055,
         0.10081829,  0.08171552],
       [ 0.02687481,  0.02216129,  0.01723541, ...,  0.01137378,
         0.02672304,  0.0736528 ]], shape=(3200, 384), dtype=float32)

Now, let's store these embeddings and metadata in a store.

In [26]:
vector_store = []

for i, chunk in enumerate(docs):
    vector_store.append({
        "text": chunk["text"],
        "source": chunk["id"],
        "embedding": np.array(embeddings[i])
    })

In [27]:
vector_store[0]

{'text': 'Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.',
 'source': 0,
 'embedding': array([ 6.98537054e-03, -6.14981093e-02, -6.68370277e-02, -8.28598905e-03,
         4.05003354e-02, -2.58229747e-02,  8.00510496e-02,  7.76855946e-02,
        -2.07008934e-03,  1.06492899e-01,  8.77064764e-02, -2.93755755e-02,
         3.29684839e-02, -2.88177598e-02,  2.90424619e-02, -2.61885524e-02,
        -2.37413638e-04, -5.43347262e-02,  5.99242747e-02, -5.19942082e-02,
         3.61571833e-02,  2.56327912e-03,  6.04414195e-02,  5.50899096e-02,
        -2.60236133e-02,  6.07446656e-02,  6.49649231e-03,  1.71788651e-02,
        -3.08122188e-02,  8.71880054e-02, -1.09186787e-02, -4.66562435e-02,
         2.76523400e-02,  5.97021692e-02, -1.20803323e-02,  1.87990796e-02,
        -2.58597266e-02, -

#### Creating a Custom Search Function

Now that our vector store is complete, let's create the search functionality

Starting with a similarity function

In [28]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

Now let's use the function for semantic search and retrieval

In [68]:
def search(query, k=3, embedding_model=embedding_fn):

    query_embedding = embedding_model.encode(query)
    
    scores = []
    
    for idx, item in enumerate(vector_store):
        sim = cosine_similarity(query_embedding, item["embedding"])
        scores.append((idx, sim))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    
    top_k = scores[:k]  # take top-k chunks
    
    return [(vector_store[idx]["text"], score) for idx, score in top_k]

In [69]:
query = "Who won the election of 1860?"

custom_results = search(query)

custom_results

[('United States presidential election, 1856', np.float32(0.68889755)),
 ('United States presidential election, 1848', np.float32(0.60962045)),
 ('On November 6, 1860, Lincoln was elected as the 16th President of the United States, beating Democrat Stephen A. Douglas, John C. Breckinridge of the Southern Democrats, and John Bell of the new Constitutional Union Party. He was the first Republican president, winning entirely on the strength of his support in the North: he was not even on the ballot in nine states in the South, and won only 2 of 996 counties in the other Southern states. Lincoln gained 1,865,908 votes (39.9% of the total), for 180 electoral votes; Douglas, 1,380,202 (29.5%) for 12 electoral votes; Breckenridge, 848,019 (18.1%) for 72 electoral votes; and Bell, 590,901 (12.5%) for 39 electoral votes. There were fusion tickets in some states, but even if his opponents had combined in every state, Lincoln had a majority vote in all but two of the states in which he won the el

Here, you can see that the desired chunk is ranked third. So relevance is not always the same as usefulness

Let's try this with a Chroma database

### Adding Data to ChromaDB

#### Creating a Chroma Collection

A simple list cannot serve as a vector store in a production RAG pipeline. Using a vectorDB like Chroma provides scalability, approximate nearest neighbour (ANN), and metadata filtering.

Let's embed and store the docs in a Chroma collection.

In [ ]:
import chromadb

chroma = chromadb.Client()

In [94]:
store = chroma.get_or_create_collection("ChromaStore")

In [95]:
# recall that we need to provide the texts and metadatas separately
texts = [doc['text'] for doc in docs]
metadatas = [str(doc['id']) for doc in docs]

store.add(
    documents = texts,
    ids = metadatas,
    # embeddings = embeddings   # can directly provide embeddings
)

In [96]:
query = "Who won the election of 1860?"

chroma_results = store.query(query_texts = [query], n_results=3)

chroma_results

{'ids': [['202', '201', '319']],
 'embeddings': None,
 'documents': [['United States presidential election, 1856',
   'United States presidential election, 1848',
   'On November 6, 1860, Lincoln was elected as the 16th President of the United States, beating Democrat Stephen A. Douglas, John C. Breckinridge of the Southern Democrats, and John Bell of the new Constitutional Union Party. He was the first Republican president, winning entirely on the strength of his support in the North: he was not even on the ballot in nine states in the South, and won only 2 of 996 counties in the other Southern states. Lincoln gained 1,865,908 votes (39.9% of the total), for 180 electoral votes; Douglas, 1,380,202 (29.5%) for 12 electoral votes; Breckenridge, 848,019 (18.1%) for 72 electoral votes; and Bell, 590,901 (12.5%) for 39 electoral votes. There were fusion tickets in some states, but even if his opponents had combined in every state, Lincoln had a majority vote in all but two of the states in

We get the same results from Chroma as well. However, we can add some conditions while querying.

In [97]:
chroma_results = store.query(query_texts = [query], n_results=3, where_document = {"$contains": "1860"})

chroma_results

{'ids': [['319', '317', '315']],
 'embeddings': None,
 'documents': [['On November 6, 1860, Lincoln was elected as the 16th President of the United States, beating Democrat Stephen A. Douglas, John C. Breckinridge of the Southern Democrats, and John Bell of the new Constitutional Union Party. He was the first Republican president, winning entirely on the strength of his support in the North: he was not even on the ballot in nine states in the South, and won only 2 of 996 counties in the other Southern states. Lincoln gained 1,865,908 votes (39.9% of the total), for 180 electoral votes; Douglas, 1,380,202 (29.5%) for 12 electoral votes; Breckenridge, 848,019 (18.1%) for 72 electoral votes; and Bell, 590,901 (12.5%) for 39 electoral votes. There were fusion tickets in some states, but even if his opponents had combined in every state, Lincoln had a majority vote in all but two of the states in which he won the electoral votes and would still have won the electoral college and the electio

This gives much better results

### Augmentation and Generation

Now that we have the final top search results, we can pass it to an LLM along with the user query and a well-engineered prompt, to generate a direct answer to the query along with citations, rather than returning whole pages/chunks.

Initialise a client first

In [52]:
import os
from dotenv import load_dotenv
from google import genai

# Load API credentials
load_dotenv(override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Initialise Client
client = genai.Client(api_key=GOOGLE_API_KEY)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Now, to pass the retrieved chunks to the LLM as context, let's join them

In [ ]:
chroma_results['documents']

['United States presidential election, 1856',
 'United States presidential election, 1848',
 'On November 6, 1860, Lincoln was elected as the 16th President of the United States, beating Democrat Stephen A. Douglas, John C. Breckinridge of the Southern Democrats, and John Bell of the new Constitutional Union Party. He was the first Republican president, winning entirely on the strength of his support in the North: he was not even on the ballot in nine states in the South, and won only 2 of 996 counties in the other Southern states. Lincoln gained 1,865,908 votes (39.9% of the total), for 180 electoral votes; Douglas, 1,380,202 (29.5%) for 12 electoral votes; Breckenridge, 848,019 (18.1%) for 72 electoral votes; and Bell, 590,901 (12.5%) for 39 electoral votes. There were fusion tickets in some states, but even if his opponents had combined in every state, Lincoln had a majority vote in all but two of the states in which he won the electoral votes and would still have won the electoral 

In [82]:
custom_context = "\n\n".join([row[0] for row in custom_results])
chroma_context = "\n\n".join(chroma_results['documents'][0])

In [89]:
def answer_query(query, context, llm = client):

    prompt = f"""Answer the question based only on the context below.
    Context: {context}
    
    Question:{query}"""

    response = client.models.generate_content(
        model = "gemini-2.5-flash-lite",
        contents = prompt
    )
    
    return response.text

So, we have defined the function to get the final LLM response for the query. Let's try it with both custom and Chroma results.

In [90]:
custom_response = answer_query(query, custom_context)

chroma_response = answer_query(query, chroma_context)

In [91]:
custom_response

'Lincoln won the election of 1860.'

In [92]:
chroma_response

'Lincoln won the election of 1860.'